In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'http://localhost:5001/'
admin_rdm_url = 'http://localhost:8001/'

# 統合管理者用 (manager)
idp_name_integrated_admin = 'FakeCAS'
idp_username_integrated_admin = None
idp_password_integrated_admin = None

# 機関管理者用 (manager)
idp_name_institutional_admin = 'FakeCAS'
idp_username_institutional_admin = None
idp_password_institutional_admin = None

# executor用（ワークフロー実行テストで使用）
idp_name_executor = 'FakeCAS'
idp_username_executor = None
idp_password_executor = None

# 別機関ユーザー用（公開範囲テストで使用）
idp_name_other_institution = 'FakeCAS'
idp_username_other_institution = None
idp_password_other_institution = None

# 機関名（統合管理者テストで使用）
institution_name = 'Massachusetts Institute of Technology [Test]'

gateway_base_url = None
workflow_zip_path = 'resources/simple-approval-workflow.zip'
default_result_path = None
close_on_fail = False
transition_timeout = 30000
skip_failed_test = False
exclude_notebooks = []
batch_project_count = 80

In [ ]:
if idp_username_integrated_admin is None:
    idp_username_integrated_admin = input(prompt=f'Username for {idp_name_integrated_admin} (integrated admin)')
if idp_password_integrated_admin is None:
    idp_password_integrated_admin = getpass(prompt=f'Password for {idp_username_integrated_admin}@{idp_name_integrated_admin}')

if idp_username_institutional_admin is None:
    idp_username_institutional_admin = input(prompt=f'Username for {idp_name_institutional_admin} (institutional admin)')
if idp_password_institutional_admin is None:
    idp_password_institutional_admin = getpass(prompt=f'Password for {idp_username_institutional_admin}@{idp_name_institutional_admin}')

if idp_username_executor is None:
    idp_username_executor = input(prompt=f'Username for {idp_name_executor} (executor)')
if idp_password_executor is None:
    idp_password_executor = getpass(prompt=f'Password for {idp_username_executor}@{idp_name_executor}')

if idp_username_other_institution is None:
    idp_username_other_institution = input(prompt=f'Username for {idp_name_other_institution} (other institution user)')
if idp_password_other_institution is None:
    idp_password_other_institution = getpass(prompt=f'Password for {idp_username_other_institution}@{idp_name_other_institution}')

if gateway_base_url is None:
    gateway_base_url = input(prompt='Workflow engine URL')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# GakuNinRDM 総合テスト [Workflowアドオン]

- サブシステム名: アドオン
- ページ/アドオン: Workflow
- 機能分類: ワークフロー管理
- シナリオ名: *
- 用意するテストデータ: URL一覧、アカウント(統合管理者: GRDM, 機関管理者: GRDM, ワークフロー実行者: GRDM)

In [ ]:
from datetime import datetime
import os
import papermill as pm
import traceback

def make_result_dir(base_path):
    result_dir = os.path.join(base_path, 'notebooks')
    os.makedirs(result_dir, exist_ok=True)
    return result_dir

def run_notebook(result_dir, base_notebook, optional_result_id=None, **optional_params):
    _, filename = os.path.split(base_notebook)
    
    if filename in exclude_notebooks:
        print(f'Skipping excluded notebook: {base_notebook}')
        return None
    
    result_id, _ = os.path.splitext(filename)
    if optional_result_id:
        result_id += optional_result_id
    result_notebook = os.path.join(result_dir, result_id + '.ipynb')
    result_path = os.path.join(result_dir, result_id)
    os.makedirs(result_path, exist_ok=True)
    params = dict(
        rdm_url=rdm_url,
        default_result_path=result_path,
        close_on_fail=True,
        transition_timeout=transition_timeout,
    )
    params.update(optional_params)
    try:
        pm.execute_notebook(
            base_notebook,
            result_notebook,
            parameters=params
        )
    except pm.PapermillExecutionError:
        if not skip_failed_test:
            raise
        print('失敗しました。テストは続行します...')
        traceback.print_exc()
    return result_notebook

result_dir = make_result_dir(default_result_path)
result_notebooks = []
result_dir

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d%H%M')

timestamp

## 統合管理者・機関管理者での一連テストの実施

エンジン登録 → テンプレート登録 → ワークフロー実行 → 公開範囲テスト → 自動有効化テスト → 有効化・無効化 の一連のテストを実施する。

- 統合管理者: 単一ユーザーテスト（プロジェクト管理者 = ワークフロー実行者）
- 機関管理者: 複数ユーザーテスト（プロジェクト管理者 ≠ ワークフロー実行者）

公開範囲テスト:
- 統合管理者: public + 別機関ユーザー → 表示される
- 機関管理者: institution + 同一機関ユーザー → 表示される、institution + 別機関ユーザー → 表示されない

自動有効化テスト:
- 統合管理者: public + 別機関ユーザー → 自動有効化される
- 機関管理者: institution + 同一機関ユーザー → 自動有効化される、institution + 別機関ユーザー → 自動有効化されない

※ publicは統合管理者のみ設定可能
※ 有効化・無効化テストでエンジンが解除されるため、公開範囲・自動有効化テストはその前に実行

In [ ]:
contexts = [
    {
        'is_integrated_admin': True,
        'suffix': '-統合管理者',
        'idp_name': idp_name_integrated_admin,
        'idp_username': idp_username_integrated_admin,
        'idp_password': idp_password_integrated_admin,
        # 単一ユーザーテスト: executor = manager
        'idp_name_executor': idp_name_integrated_admin,
        'idp_username_executor': idp_username_integrated_admin,
        'idp_password_executor': idp_password_integrated_admin,
    },
    {
        'is_integrated_admin': False,
        'suffix': '-機関管理者',
        'idp_name': idp_name_institutional_admin,
        'idp_username': idp_username_institutional_admin,
        'idp_password': idp_password_institutional_admin,
        # 複数ユーザーテスト: executor = testuser2
        'idp_name_executor': idp_name_executor,
        'idp_username_executor': idp_username_executor,
        'idp_password_executor': idp_password_executor,
    },
]

last_workflow_name = None
last_idp_params = None
last_template_project = None
last_engine_name = None

for ctx in contexts:
    suffix = ctx['suffix']
    is_integrated_admin = ctx['is_integrated_admin']
    idp_params = dict(
        idp_name_1=ctx['idp_name'],
        idp_username_1=ctx['idp_username'],
        idp_password_1=ctx['idp_password'],
    )
    
    engine_name = f'TEST-ENGINE-{timestamp}{suffix}'
    project_name_template = f'TEST-WORKFLOW-TEMPLATE-{timestamp}{suffix}'
    project_name_execution = f'TEST-WORKFLOW-EXEC-{timestamp}{suffix}'
    workflow_name = f'TEST-TEMPLATE-{timestamp}{suffix}'
    last_workflow_name = workflow_name
    last_idp_params = idp_params
    last_template_project = project_name_template
    last_engine_name = engine_name
    
    # エンジン登録
    result_notebooks.append(run_notebook(
        result_dir,
        'テスト手順-Workflowアドオン-エンジン登録.ipynb',
        suffix,
        project_name=project_name_template,
        admin_rdm_url=admin_rdm_url,
        engine_name=engine_name,
        gateway_base_url=gateway_base_url,
        institution_name=institution_name,
        is_integrated_admin=is_integrated_admin,
        **idp_params,
    ))
    
    # テンプレート登録
    result_notebooks.append(run_notebook(
        result_dir,
        'テスト手順-Workflowアドオン-ワークフローテンプレート登録.ipynb',
        suffix,
        project_name=project_name_template,
        engine_name=engine_name,
        workflow_zip_path=workflow_zip_path,
        workflow_name=workflow_name,
        **idp_params,
    ))
    
    # ワークフロー実行
    result_notebooks.append(run_notebook(
        result_dir,
        'テスト手順-Workflowアドオン-ワークフロー実行.ipynb',
        suffix,
        project_name=project_name_execution,
        workflow_name=workflow_name,
        idp_name_2=ctx['idp_name_executor'],
        idp_username_2=ctx['idp_username_executor'],
        idp_password_2=ctx['idp_password_executor'],
        **idp_params,
    ))
    
    # 公開範囲テスト
    if is_integrated_admin:
        # public + 別機関ユーザー → 表示される
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-公開範囲の設定.ipynb',
            '-public-別機関',
            template_project_name=project_name_template,
            engine_name=engine_name,
            workflow_zip_path=workflow_zip_path,
            idp_name_registrar=ctx['idp_name'],
            idp_username_registrar=ctx['idp_username'],
            idp_password_registrar=ctx['idp_password'],
            visibility='public',
            idp_name_test=idp_name_other_institution,
            idp_username_test=idp_username_other_institution,
            idp_password_test=idp_password_other_institution,
            expect_visible=True,
        ))
    else:
        # institution + 同一機関ユーザー → 表示される
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-公開範囲の設定.ipynb',
            '-institution-同一機関',
            template_project_name=project_name_template,
            engine_name=engine_name,
            workflow_zip_path=workflow_zip_path,
            idp_name_registrar=ctx['idp_name'],
            idp_username_registrar=ctx['idp_username'],
            idp_password_registrar=ctx['idp_password'],
            visibility='institution',
            idp_name_test=idp_name_executor,
            idp_username_test=idp_username_executor,
            idp_password_test=idp_password_executor,
            expect_visible=True,
        ))
        
        # institution + 別機関ユーザー → 表示されない
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-公開範囲の設定.ipynb',
            '-institution-別機関',
            template_project_name=project_name_template,
            engine_name=engine_name,
            workflow_zip_path=workflow_zip_path,
            idp_name_registrar=ctx['idp_name'],
            idp_username_registrar=ctx['idp_username'],
            idp_password_registrar=ctx['idp_password'],
            visibility='institution',
            idp_name_test=idp_name_other_institution,
            idp_username_test=idp_username_other_institution,
            idp_password_test=idp_password_other_institution,
            expect_visible=False,
        ))
    
    # 自動有効化テスト
    if is_integrated_admin:
        # public + 別機関ユーザー → 自動有効化される
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-自動有効化.ipynb',
            '-public-別機関',
            template_project_name=project_name_template,
            engine_name=engine_name,
            workflow_zip_path=workflow_zip_path,
            idp_name_registrar=ctx['idp_name'],
            idp_username_registrar=ctx['idp_username'],
            idp_password_registrar=ctx['idp_password'],
            visibility='public',
            idp_name_test=idp_name_other_institution,
            idp_username_test=idp_username_other_institution,
            idp_password_test=idp_password_other_institution,
            expect_auto_activate=True,
        ))
    else:
        # institution + 同一機関ユーザー → 自動有効化される
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-自動有効化.ipynb',
            '-institution-同一機関',
            template_project_name=project_name_template,
            engine_name=engine_name,
            workflow_zip_path=workflow_zip_path,
            idp_name_registrar=ctx['idp_name'],
            idp_username_registrar=ctx['idp_username'],
            idp_password_registrar=ctx['idp_password'],
            visibility='institution',
            idp_name_test=idp_name_executor,
            idp_username_test=idp_username_executor,
            idp_password_test=idp_password_executor,
            expect_auto_activate=True,
        ))
        
        # institution + 別機関ユーザー → 自動有効化されない
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-自動有効化.ipynb',
            '-institution-別機関',
            template_project_name=project_name_template,
            engine_name=engine_name,
            workflow_zip_path=workflow_zip_path,
            idp_name_registrar=ctx['idp_name'],
            idp_username_registrar=ctx['idp_username'],
            idp_password_registrar=ctx['idp_password'],
            visibility='institution',
            idp_name_test=idp_name_other_institution,
            idp_username_test=idp_username_other_institution,
            idp_password_test=idp_password_other_institution,
            expect_auto_activate=False,
        ))
    
    # 有効化・無効化（統合管理者のみ）
    if is_integrated_admin:
        result_notebooks.append(run_notebook(
            result_dir,
            'テスト手順-Workflowアドオン-有効化・無効化.ipynb',
            suffix,
            project_name_template=project_name_template,
            project_name_execution=project_name_execution,
            workflow_name=workflow_name,
            engine_name=engine_name,
            admin_rdm_url=admin_rdm_url,
            institution_name=institution_name,
            workflow_zip_path=workflow_zip_path,
            **idp_params,
        ))

result_notebooks

## 「ワークフロー実行（creator承認）」テストの実施

3者（executor, manager, creator）によるワークフロー実行と承認のテストを実施する。

- creator: 機関管理者（テンプレート登録者）
- manager: executor用ユーザー（プロジェクト管理者）
- executor: 別機関ユーザー（申請者）

※ テンプレート登録プロジェクトは機関管理者テストで作成済みのものを使用
※ ワークフローZIPはcreator承認用のものを使用

In [ ]:
result_notebooks.append(run_notebook(
    result_dir,
    'テスト手順-Workflowアドオン-ワークフロー実行（creator承認）.ipynb',
    template_project_name=last_template_project,
    engine_name=last_engine_name,
    workflow_zip_path='resources/creator-approval-workflow.zip',
    # creator: 機関管理者
    idp_name_creator=idp_name_institutional_admin,
    idp_username_creator=idp_username_institutional_admin,
    idp_password_creator=idp_password_institutional_admin,
    # manager: executor用ユーザー
    idp_name_manager=idp_name_executor,
    idp_username_manager=idp_username_executor,
    idp_password_manager=idp_password_executor,
    # executor: 別機関ユーザー
    idp_name_executor=idp_name_other_institution,
    idp_username_executor=idp_username_other_institution,
    idp_password_executor=idp_password_other_institution,
))
result_notebooks[-1]

## 「ワークフロー一括実行」テストの実施

テスト「テスト手順-Workflowアドオン-ワークフロー一括実行」を実施する。

In [ ]:
result_notebooks.append(run_notebook(
    result_dir,
    'テスト手順-Workflowアドオン-ワークフロー一括実行.ipynb',
    project_count=batch_project_count,
    workflow_name=last_workflow_name,
    **last_idp_params,
))
result_notebooks[-1]

## 後始末

In [ ]:
!rm -fr {work_dir}